# Вариант 2. CDI → ЕГРН: помещение или здание

`full_address` передаётся в CDI без изменений.

Порядок поиска:

1. точное помещение по ФИАС квартиры или помещения;
2. единственное помещение в доме с совпадающей площадью;
3. здание по ФИАС дома.

Неоднозначные связи не присоединяются.


In [1]:
%pip install pandas sqlalchemy "psycopg[binary]" oracledb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
from pathlib import Path
import oracledb
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [3]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks' / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR / 'notebooks'
else:
    NOTEBOOK_DIR = CURRENT_DIR

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == 'notebooks'
    else NOTEBOOK_DIR
)
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)
print('Папка результатов:', OUTPUT_DIR)

Корень проекта: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python
Папка результатов: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО


# 1. Подключение к Сфере


In [4]:
CREDENTIALS_PATH = NOTEBOOK_DIR / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')


Учётные данные прочитаны, подключение к Сфере создано


In [5]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

,database_name,user_name
0,postgres,svetovavs


# 2. Подключение к Oracle КХД



In [6]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')


Подключение к КХД создано


In [7]:
# проверяем доступ к ЕГРН
khd_schema_for_check = KHD_DATA_SCHEMA.upper()
if not khd_schema_for_check.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

with khd_connection.cursor() as cursor:
    cursor.execute(
        f'select 1 from {khd_schema_for_check}.EGRN_DATA '
        'where rownum = 1'
    )
    cursor.fetchone()

print('Таблица EGRN_DATA доступна')


Таблица EGRN_DATA доступна


# 3. SQL Сфера, расширенный подход


In [8]:
expanded_sql = r"""
/*
запускать в сфере

запрос собирает все неудаленные объекты недвижимости
если объект связан с подходящим договором данные договора заполняются
если связь не найдена объект остается в результате с пустыми полями договора

одна строка для связанного объекта означает объект в одном договоре
одна строка для несвязанного объекта означает его последнюю версию характеристик
*/

with task_candidates as (
    /* отбираем подходящие задачи оформления */
    select
        t.id as task_id,
        r.id as request_id,
        c.id as contract_id,
        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* оставляем последнюю подходящую задачу каждого договора */
    select
        task_id,
        request_id,
        contract_id
    from task_candidates
    where task_number = 1
),

linked_object_candidates as (
    /* находим недвижимость в выбранных задачах */
    select
        ch.insurance_object_id as object_id,
        ch.id as characteristics_id,
        link.id as task_object_link_id,
        selected.task_id,
        selected.request_id,
        selected.contract_id,
        row_number() over (
            partition by selected.task_id, ch.insurance_object_id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc
        ) as link_number
    from selected_tasks selected
    join bps_request_ins_task_insurance_object link
        on link.parent_id = selected.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_links as (
    /* убираем повторные связи одного объекта с одной задачей */
    select
        object_id,
        characteristics_id,
        task_object_link_id,
        task_id,
        request_id,
        contract_id
    from linked_object_candidates
    where link_number = 1
),

object_versions as (
    /* нумеруем версии характеристик каждого объекта */
    select
        obj.id as object_id,
        ch.id as characteristics_id,
        row_number() over (
            partition by obj.id
            order by
                ch.version_is_active desc nulls last,
                ch.version_number desc nulls last,
                ch.version_start_date desc nulls last,
                ch.id desc nulls last
        ) as version_number
    from base_insurance_object obj
    left join base_insurance_object_characteristics ch
        on ch.insurance_object_id = obj.id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

dataset_keys as (
    /* сохраняем все найденные связи с договорами */
    select
        linked.object_id,
        linked.characteristics_id,
        linked.task_object_link_id,
        linked.task_id,
        linked.request_id,
        linked.contract_id,
        'linked'::text as row_source
    from selected_links linked

    union all

    /* добавляем объекты для которых подходящий договор не найден */
    select
        version.object_id,
        version.characteristics_id,
        null::integer as task_object_link_id,
        null::integer as task_id,
        null::integer as request_id,
        null::integer as contract_id,
        'not_linked'::text as row_source
    from object_versions version
    where version.version_number = 1
      and not exists (
          select 1
          from selected_links linked
          where linked.object_id = version.object_id
      )
),

object_link_profile as (
    /* считаем со сколькими договорами связан объект */
    select
        object_id,
        count(distinct contract_id) as contract_count
    from selected_links
    group by object_id
),

selected_characteristics as (
    /* ограничиваем расчет условий версиями из итоговой выборки */
    select distinct characteristics_id
    from dataset_keys
    where characteristics_id is not null
),

condition_summary as (
    /* сворачиваем варианты условий в одну строку */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        count(cond.insured_sum) as filled_insured_sum_count,
        count(distinct cond.insured_sum) filter (
            where cond.insured_sum is not null
        ) as distinct_insured_sum_count,
        min(cond.insured_sum) as minimum_insured_sum,
        max(cond.insured_sum) as maximum_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        array_agg(
            distinct cond.terms_option_number
            order by cond.terms_option_number
        ) filter (
            where cond.terms_option_number is not null
        ) as terms_option_numbers,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        case
            when count(distinct cond.insured_sum) filter (
                where cond.insured_sum is not null
            ) = 1
             and count(distinct cond.insured_sum_currency) filter (
                where cond.insured_sum_currency is not null
            ) <= 1
            then max(cond.insured_sum)
        end as insured_sum
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

raw_result as (
/* собираем исходные поля расширенного датасета */
select
    /* качество строки */
    keys.row_source,
    case
        when coalesce(profile.contract_count, 0) = 0 then 'not_linked'
        when profile.contract_count = 1 then 'linked'
        else 'multiple_contracts'
    end as contract_link_status,
    coalesce(profile.contract_count, 0) as contract_count,
    (keys.contract_id is not null) as has_contract,
    (address.id is not null) as has_address,
    (conditions.insured_sum is not null) as has_target,
    case
        when conditions.condition_count is null then 'no_conditions'
        when conditions.filled_insured_sum_count = 0 then 'target_is_empty'
        when conditions.distinct_insured_sum_count > 1 then 'several_target_values'
        when conditions.currency_count > 1 then 'several_currencies'
        when conditions.insured_sum <= 0 then 'target_is_not_positive'
        else 'target_is_usable'
    end as target_status,

    /* идентификаторы */
    keys.object_id,
    keys.characteristics_id,
    keys.task_object_link_id,
    keys.task_id,
    keys.request_id,
    keys.contract_id,
    obj.geo_address_id,
    contract.contractor_id as policyholder_id,
    request.corporate_crm_id,

    /* целевая страховая сумма */
    conditions.insured_sum,
    conditions.insured_sum_currency,
    conditions.condition_count,
    conditions.filled_insured_sum_count,
    conditions.distinct_insured_sum_count,
    conditions.minimum_insured_sum as condition_min_insured_sum,
    conditions.maximum_insured_sum as condition_max_insured_sum,
    conditions.currency_count as condition_currency_count,
    conditions.terms_option_numbers,

    /* контрольные суммы */
    task_link.insured_sum as task_object_insured_sum,
    task_link.insured_sum_currency as task_object_insured_sum_currency,
    task.total_ins_contract_amount as contract_insured_sum,
    task.curr_ins_contract_amount as contract_amount_currency,
    task.total_ins_contract_premium as contract_premium,
    ch.insurance_value,
    ch.insurance_value_currency,
    ch.insurance_value_basis,
    ch.pledged_value,
    conditions.minimum_per_occurrence_limit,
    conditions.maximum_per_occurrence_limit,

    /* объект */
    obj.obj_type as object_type,
    obj.elementary_obj_type,
    obj.obj_name as object_name,
    obj.description as object_description,
    obj.original_address,
    obj.active as object_is_active,
    obj.d_create as object_create_date,
    obj.d_change as object_change_date,

    /* характеристики объекта */
    ch.version_number as characteristics_version_number,
    ch.version_start_date as characteristics_version_start_date,
    ch.version_end_date as characteristics_version_end_date,
    ch.version_is_active as characteristics_version_is_active,
    ch.ownership_type,
    ch.is_pledged,
    ch.is_leased,
    ch.insured_components,
    ch.activity_types,
    ch.risk_natures,
    ch.insurance_territory,
    ch.has_losses,
    ch.insurance_object_loss_history,
    ch.characteristics ->> 'total_area_sq_m' as total_area,
    ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
    ch.characteristics ->> 'construction_year' as construction_year,
    ch.characteristics ->> 'last_capital_repair_year' as capital_repair_year,
    ch.characteristics ->> 'total_floors_count' as floors_count,
    ch.characteristics ->> 'occupied_floor' as occupied_floor,
    ch.characteristics ->> 'load_bearing_walls_material' as walls_material,
    ch.characteristics ->> 'interfloor_overlap_material' as overlap_material,
    ch.characteristics ->> 'roofing_material' as roofing_material,
    ch.characteristics ->> 'fire_alarm_system_availability'
        as fire_alarm_system_availability,
    ch.characteristics ->> 'fire_suppression_system_availability'
        as fire_suppression_system_availability,
    ch.characteristics ->> 'nearest_fire_station_distance_km'
        as nearest_fire_station_distance_km,
    ch.characteristics as object_characteristics_json,

    /* адрес */
    address.full_address,
    address.postal_code,
    address.region_id as address_region_id,
    address.area as district,
    address.settlement_type,
    address.settlement,
    address.street_type,
    address.street,
    address.house,
    address.building,
    address.block,
    address.flat,
    address.office,
    address.fias_code,
    address.longitude,
    address.latitude,
    address.address_dgis_id,

    /* договор */
    contract.n_contract as contract_number,
    contract.document_status as contract_status,
    contract.system_type as contract_source_system,
    contract.ins_product_sbs as insurance_product,
    contract.d_sign_contract as contract_sign_date,
    contract.d_start_contract as contract_start_date,
    contract.d_end_contract as contract_end_date,
    contract.prevcontract_id as previous_contract_id,
    contract.rootcontract_id as root_contract_id,
    previous_contract.n_contract as previous_contract_number,
    previous_contract.d_start_contract as previous_contract_start_date,
    previous_contract.d_end_contract as previous_contract_end_date,

    /* задача и заявка */
    task.task_type,
    task.status as task_status,
    task.ins_document_type,
    task.ins_refuse,
    task.d_create as task_create_date,
    task.d_conclusion_ins_contract as contract_conclusion_date,
    task.ins_product as task_product,
    task.industry as task_industry,
    task.subindustry as task_subindustry,
    task.locations_count,
    task.multi_location,
    task.object_description as task_object_description,
    request.business_segment,
    request.sale_channel,
    request.ins_product as request_product,

    /* страхователь и crm */
    policyholder.inn as policyholder_inn,
    policyholder.company_name_short as policyholder_name,
    policyholder.cdi_id as policyholder_cdi_id,
    crm.segment as crm_segment,
    crm.macroindustry as crm_macroindustry,
    crm.industry as crm_industry,
    crm.okved as crm_okved,

    /* дата состояния строки */
    coalesce(
        task.d_conclusion_ins_contract::timestamp with time zone,
        contract.d_sign_contract,
        ch.version_start_date,
        obj.d_create
    ) as as_of_date
from dataset_keys keys
join base_insurance_object obj
    on obj.id = keys.object_id
left join base_insurance_object_characteristics ch
    on ch.id = keys.characteristics_id
left join condition_summary conditions
    on conditions.characteristics_id = keys.characteristics_id
left join bps_request_ins_task_insurance_object task_link
    on task_link.id = keys.task_object_link_id
left join bps_request_ins_task task
    on task.id = keys.task_id
left join bps_request_ins request
    on request.id = keys.request_id
left join bps_contract contract
    on contract.id = keys.contract_id
left join bps_contract previous_contract
    on previous_contract.id = contract.prevcontract_id
left join bps_contractor policyholder
    on policyholder.id = contract.contractor_id
left join bps_corporate_crm crm
    on crm.id = request.corporate_crm_id
left join base_geo_address address
    on address.id = obj.geo_address_id
left join object_link_profile profile
    on profile.object_id = keys.object_id
),

standardized_result as (
    /* приводим результат к общей структуре двух датасетов */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
            or nullif(btrim(raw.original_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse
    from raw_result raw
)

select *
from standardized_result
order by
    has_contract desc,
    as_of_date desc nulls last,
    object_id;

"""


In [9]:
with engine.connect() as connection:
    expanded_df = pd.read_sql_query(text(expanded_sql), connection)

print('Строк:', len(expanded_df))
print('Колонок:', len(expanded_df.columns))
display(expanded_df.head(3))


Строк: 14692
Колонок: 102


,row_source,contract_link_status,contract_count,has_contract,has_address,has_target,target_status,contract_id,contract_number,previous_contract_id,root_contract_id,request_id,task_id,task_object_link_id,characteristics_id,object_id,geo_address_id,policyholder_id,corporate_crm_id,as_of_date,contract_conclusion_date,contract_sign_date,contract_start_date,contract_end_date,contract_status,ins_document_type,insurance_product,real_estate_objects_in_contract,object_name,object_description,object_type,elementary_obj_type,total_area,occupied_area,construction_year,capital_repair_year,floors_count,occupied_floor,walls_material,overlap_material,roofing_material,ownership_type,is_leased,insured_components,activity_types,risk_natures,insurance_territory,full_address,original_address,postal_code,...,settlement,street,house,building,block,flat,office,fias_code,longitude,latitude,address_dgis_id,policyholder_inn,policyholder_name,policyholder_cdi_id,crm_segment,crm_macroindustry,crm_industry,crm_okved,business_segment,task_industry,task_subindustry,contract_insured_sum,contract_amount_currency,task_object_insured_sum,task_object_insured_sum_currency,condition_min_insured_sum,condition_max_insured_sum,insured_sum,insured_sum_currency,condition_currency_count,contract_premium,insurance_value,insurance_value_currency,insurance_value_basis,is_pledged,pledged_value,minimum_per_occurrence_limit,maximum_per_occurrence_limit,previous_contract_number,previous_contract_start_date,previous_contract_end_date,condition_count,characteristics_version_number,characteristics_version_start_date,characteristics_version_end_date,characteristics_version_is_active,object_characteristics_json,task_type,task_status,ins_refuse
0,linked,linked,1,True,True,True,target_is_usable,117291482.0,013БС4040041483,None,None,436812.0,699424.0,36737.0,24498,24520,10104.0,19412267.0,102090.0,2026-08-27 21:00:00+00:00,2026-08-28,2025-08-27 21:00:00+00:00,2025-09-14 21:00:00+00:00,2026-09-14 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,СББ. Имущество юридических лиц (залоги) - свер...,5.0,warehouse,Нежилое здание (склад №1),prop_assets_legal_entities,nedv_ul_and_ip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,"[engineering_equipment, structural_elements]",[],[],None,"Садовый, Пасечная улица, 11/1 к1",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,82.985386,55.142120,141373143687001,5402553389,"ООО ""НЛК2""",None,ksb,NaN,NaN,NaN,ksb,warehouse_risks,cooled_and_frozen_products_warehouse,9.225150e+08,RUB,NaN,NaN,3.256785e+08,3.256785e+08,3.256785e+08,RUB,1.0,1436448.13,3.256785e+08,RUB,NaN,True,None,NaN,NaN,None,None,None,1.0,1,2026-07-23 10:47:46.800048+00:00,None,True,{},draft_contract,operational_archive,False
1,linked,linked,1,True,True,True,target_is_usable,117291482.0,013БС4040041483,None,None,436812.0,699424.0,36738.0,24499,24521,13181.0,19412267.0,102090.0,2026-08-27 21:00:00+00:00,2026-08-28,2025-08-27 21:00:00+00:00,2025-09-14 21:00:00+00:00,2026-09-14 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,СББ. Имущество юридических лиц (залоги) - свер...,5.0,warehouse,Нежилое здание (склад №2),prop_assets_legal_entities,nedv_ul_and_ip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,"[engineering_equipment, structural_elements]",[],[],None,"Садовый, Пасечная улица, 11/1 к2",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,82.985701,55.141124,141373143687002,5402553389,"ООО ""НЛК2""",None,ksb,NaN,NaN,NaN,ksb,warehouse_risks,cooled_and_frozen_products_warehouse,9.225150e+08,RUB,NaN,NaN,3.043775e+08,3.043775e+08,3.043775e+08,RUB,1.0,1436448.13,3.043775e+08,RUB,NaN,True,None,NaN,NaN,None,None,None,1.0,1,2026-07-23 10:50:15.025573+00:00,None,True,{},draft_contract,operational_archive,False
2,linked,linked,1,True,True,True,target_is_usable,117291482.0,013БС4040041483,None,None,436812.0,699424.0,36739.0,24500,24522,13182.0,19412267.0,102090.0,2026-08-27 21:00:00+00:00,2026-08-28,2025-08-27 21:00:00+00:00,2025-09-14 21:00:00+00:00,2026-09-14 20:59:59.999999+00:00,Оформлен,ins_contract_p

# 4. Проверка заполненности

In [10]:
required_columns = {
    'object_id', 'characteristics_id', 'elementary_obj_type',
    'insured_sum', 'full_address', 'total_area', 'row_source'
}
missing_columns = sorted(required_columns - set(expanded_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных объектов',
        'Строк с договором',
        'Строк без договора',
        'Строк с target',
        'Строк с full_address',
    ],
    'Значение': [
        len(expanded_df),
        expanded_df['object_id'].nunique(dropna=True),
        expanded_df['has_contract'].fillna(False).sum(),
        (~expanded_df['has_contract'].fillna(False)).sum(),
        expanded_df['insured_sum'].notna().sum(),
        expanded_df['full_address'].fillna('').str.strip().ne('').sum(),
    ],
})
display(profile)


,Показатель,Значение
0,Строк,14692
1,Уникальных объектов,14682
2,Строк с договором,1211
3,Строк без договора,13481
4,Строк с target,13507
5,Строк с full_address,11733


In [11]:
display(expanded_df['row_source'].fillna('empty').value_counts(dropna=False))
display(expanded_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))


row_source
not_linked    13481
linked         1211
Name: count, dtype: int64

elementary_obj_type
nedv_ul_and_ip    14692
Name: count, dtype: int64

# 5. Получение ФИАС через CDI

Кждый уникальный `full_address` передаётся в CDI без изменений. CDI возвращает ФИАС дома и, если адрес достаточно точный, ФИАС помещения.

Адрес разбирает сам CDI. Ноутбук не удаляет квартиру, не меняет регистр и не переставляет части адреса.


In [12]:
# готовим уникальные full_address для поиска CDI
sphere_with_row_id = expanded_df.copy()
sphere_with_row_id.insert(
    0,
    'sphere_row_id',
    range(1, len(sphere_with_row_id) + 1),
)
sphere_with_row_id['source_address'] = (
    sphere_with_row_id['full_address'].astype('string')
)
empty_address = (
    sphere_with_row_id['source_address'].str.strip().eq('')
)
sphere_with_row_id.loc[empty_address, 'source_address'] = pd.NA

unique_addresses = (
    sphere_with_row_id[['source_address']]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_addresses.insert(
    0,
    'address_lookup_id',
    range(1, len(unique_addresses) + 1),
)

address_records = [
    {
        'address_lookup_id': int(row.address_lookup_id),
        'full_address': str(row.source_address),
    }
    for row in unique_addresses.itertuples(index=False)
]
address_json = json.dumps(address_records, ensure_ascii=False)

print('Объектов:', len(sphere_with_row_id))
print('Объектов с full_address:', sphere_with_row_id['source_address'].notna().sum())
print('Уникальных full_address:', len(unique_addresses))


Объектов: 14692
Объектов с full_address: 11733
Уникальных full_address: 7512


In [13]:
# full_address передаётся в функцию CDI без изменений
CDI_SCHEMA = credentials.get('CDI_SCHEMA', 'DM_MOTOR').upper()
CDI_TEXT_FUNCTION = credentials.get(
    'CDI_TEXT_FUNCTION',
    'F_GET_CDI_ADDR_BY_TEXT',
).upper()

for value, label in [
    (CDI_SCHEMA, 'CDI_SCHEMA'),
    (CDI_TEXT_FUNCTION, 'CDI_TEXT_FUNCTION'),
]:
    if not value.replace('_', '').isalnum():
        raise ValueError(f'Некорректное значение {label}')

cdi_function_candidates = [
    f'select d.* from {CDI_SCHEMA}.{CDI_TEXT_FUNCTION}(:address_text) d',
    f'select d.* from table({CDI_SCHEMA}.{CDI_TEXT_FUNCTION}(:address_text)) d',
    f'select d.* from {CDI_TEXT_FUNCTION}(:address_text) d',
    f'select d.* from table({CDI_TEXT_FUNCTION}(:address_text)) d',
]

# в Oracle форма вызова функции может зависеть от версии и прав
cdi_function_sql = cdi_function_candidates[0]
if not unique_addresses.empty:
    test_address = unique_addresses.iloc[0]['source_address']
    call_errors = []
    cdi_function_sql = None
    with khd_connection.cursor() as cursor:
        for candidate_sql in cdi_function_candidates:
            try:
                cursor.execute(candidate_sql, address_text=test_address)
                cursor.fetchmany(1)
                cdi_function_sql = candidate_sql
                break
            except oracledb.Error as error:
                call_errors.append(str(error))

    if cdi_function_sql is None:
        error_details = '\n'.join(
            f'{number}. {message}'
            for number, message in enumerate(call_errors, start=1)
        )
        raise RuntimeError(
            'Не удалось вызвать CDI. '
            f'Подключение: user={KHD_USER}, service={KHD_SERVICE_NAME}.\n'
            f'Ошибки Oracle:\n{error_details}'
        )

print('Вызов CDI:', cdi_function_sql)

cdi_raw_records = []
cdi_error_records = []
cdi_result_columns = None

with khd_connection.cursor() as cursor:
    for row_number, row in enumerate(
        unique_addresses.itertuples(index=False),
        start=1,
    ):
        try:
            cursor.execute(
                cdi_function_sql,
                address_text=row.source_address,
            )
            result_columns = [
                str(column[0]).lower()
                for column in cursor.description
            ]
            if cdi_result_columns is None:
                cdi_result_columns = result_columns

            while True:
                batch = cursor.fetchmany(100)
                if not batch:
                    break
                for values in batch:
                    record = dict(zip(result_columns, values))
                    record['address_lookup_id'] = int(row.address_lookup_id)
                    record['sphere_full_address'] = row.source_address
                    cdi_raw_records.append(record)

        except oracledb.Error as error:
            error_text = str(error)
            cdi_error_records.append({
                'address_lookup_id': int(row.address_lookup_id),
                'error': error_text,
            })

            # такая ошибка означает, что функция недоступна
            if any(code in error_text for code in [
                'ORA-00904', 'ORA-00942', 'ORA-06550', 'ORA-01031'
            ]):
                raise RuntimeError(
                    'Функция CDI недоступна: '
                    f'{CDI_SCHEMA}.{CDI_TEXT_FUNCTION}. '
                    'Ошибка Oracle: ' + error_text
                ) from error

        if row_number % 100 == 0:
            print(
                'Обработано адресов:',
                row_number,
                'из',
                len(unique_addresses),
            )

cdi_raw_df = pd.DataFrame(cdi_raw_records)
cdi_errors_df = pd.DataFrame(cdi_error_records)

# если адресов нет, сохраняем объекты без связи с CDI
if unique_addresses.empty:
    cdi_result_columns = ['house_fias_id']

if cdi_result_columns is None:
    raise RuntimeError('CDI не вернул структуру результата')

# в разных версиях CDI колонка ФИАС дома может называться по-разному
house_fias_column = next(
    (
        column
        for column in [
            'house_fias_id',
            'fias_id_house',
            'fias_house_id',
        ]
        if column in cdi_result_columns
    ),
    None,
)
if house_fias_column is None:
    raise ValueError(
        'CDI не вернул колонку ФИАС дома. '
        'Колонки: ' + ', '.join(cdi_result_columns)
    )

flat_fias_column = next(
    (
        column
        for column in [
            'flat_fias_id',
            'fias_id_flat',
            'fias_flat_id',
        ]
        if column in cdi_result_columns
    ),
    None,
)

if flat_fias_column is None:
    flat_fias_column = 'flat_fias_id'
    cdi_raw_df[flat_fias_column] = pd.NA

if cdi_raw_df.empty:
    cdi_raw_df = pd.DataFrame(
        columns=[
            'address_lookup_id',
            'sphere_full_address',
            house_fias_column,
            flat_fias_column,
        ]
    )

cdi_raw_df['house_fias_for_match'] = (
    cdi_raw_df[house_fias_column]
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

cdi_raw_df['flat_fias_for_match'] = (
    cdi_raw_df[flat_fias_column]
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

cdi_summary = (
    cdi_raw_df.groupby('address_lookup_id', dropna=False)
    .agg(
        cdi_address_candidate_count=('address_lookup_id', 'size'),
        cdi_house_fias_candidate_count=(
            'house_fias_for_match',
            'nunique',
        ),
        cdi_flat_fias_candidate_count=(
            'flat_fias_for_match',
            'nunique',
        ),
    )
    .reset_index()
)

unique_fias = (
    cdi_summary['cdi_house_fias_candidate_count'].eq(1)
)
unique_address_ids = set(
    cdi_summary.loc[unique_fias, 'address_lookup_id']
)
cdi_chosen = (
    cdi_raw_df.loc[
        cdi_raw_df['address_lookup_id'].isin(unique_address_ids)
        & cdi_raw_df['house_fias_for_match'].notna()
    ]
    .drop_duplicates(['address_lookup_id', 'house_fias_for_match'])
    .drop_duplicates('address_lookup_id')
    [['address_lookup_id', 'house_fias_for_match']]
    .rename(columns={'house_fias_for_match': 'cdi_fias_id_house'})
)

unique_flat_ids = set(
    cdi_summary.loc[
        cdi_summary['cdi_flat_fias_candidate_count'].eq(1),
        'address_lookup_id',
    ]
)
cdi_flat_chosen = (
    cdi_raw_df.loc[
        cdi_raw_df['address_lookup_id'].isin(unique_flat_ids)
        & cdi_raw_df['flat_fias_for_match'].notna()
    ]
    .drop_duplicates(['address_lookup_id', 'flat_fias_for_match'])
    .drop_duplicates('address_lookup_id')
    [['address_lookup_id', 'flat_fias_for_match']]
    .rename(columns={'flat_fias_for_match': 'cdi_fias_id_flat'})
)

cdi_lookup_df = (
    unique_addresses
    .rename(columns={'source_address': 'sphere_full_address'})
    .merge(cdi_summary, on='address_lookup_id', how='left')
    .merge(cdi_chosen, on='address_lookup_id', how='left')
    .merge(cdi_flat_chosen, on='address_lookup_id', how='left')
)

for column in [
    'cdi_address_candidate_count',
    'cdi_house_fias_candidate_count',
    'cdi_flat_fias_candidate_count',
]:
    cdi_lookup_df[column] = (
        cdi_lookup_df[column].fillna(0).astype('int64')
    )

cdi_lookup_df['cdi_is_unique_match'] = (
    cdi_lookup_df['cdi_house_fias_candidate_count'].eq(1).astype('int64')
)
cdi_lookup_df['cdi_match_status'] = 'not_found'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_house_fias_candidate_count'].eq(1),
    'cdi_match_status',
] = 'unique_house_fias'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_house_fias_candidate_count'].gt(1),
    'cdi_match_status',
] = 'ambiguous_house_fias'

# ошибку вызова CDI не путаем с отсутствием адреса
if not cdi_errors_df.empty:
    error_address_ids = set(cdi_errors_df['address_lookup_id'])
    error_mask = cdi_lookup_df['address_lookup_id'].isin(
        error_address_ids
    )
    cdi_lookup_df.loc[error_mask, 'cdi_match_status'] = 'lookup_error'
    cdi_lookup_df.loc[error_mask, 'cdi_is_unique_match'] = 0
    cdi_lookup_df.loc[error_mask, 'cdi_fias_id_house'] = pd.NA

cdi_lookup_df['cdi_flat_is_unique_match'] = (
    cdi_lookup_df['cdi_flat_fias_candidate_count'].eq(1).astype('int64')
)
cdi_lookup_df['cdi_flat_match_status'] = 'not_found'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_flat_fias_candidate_count'].eq(1),
    'cdi_flat_match_status',
] = 'unique_flat_fias'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_flat_fias_candidate_count'].gt(1),
    'cdi_flat_match_status',
] = 'ambiguous_flat_fias'

cdi_lookup_df['cdi_match_level'] = pd.NA
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_is_unique_match'].eq(1),
    'cdi_match_level',
] = 'full_address_via_cdi'

print('Адресов передано в CDI:', len(unique_addresses))
print('Адресов с одним ФИАС дома:', cdi_lookup_df['cdi_is_unique_match'].sum())
print('Ошибок обработки адреса:', len(cdi_errors_df))
print(cdi_lookup_df['cdi_match_status'].value_counts(dropna=False))


Вызов CDI: select d.* from DM_MOTOR.F_GET_CDI_ADDR_BY_TEXT(:address_text) d
Обработано адресов: 100 из 7512
Обработано адресов: 200 из 7512
Обработано адресов: 300 из 7512
Обработано адресов: 400 из 7512
Обработано адресов: 500 из 7512
Обработано адресов: 600 из 7512
Обработано адресов: 700 из 7512
Обработано адресов: 800 из 7512
Обработано адресов: 900 из 7512
Обработано адресов: 1000 из 7512
Обработано адресов: 1100 из 7512
Обработано адресов: 1200 из 7512
Обработано адресов: 1300 из 7512
Обработано адресов: 1400 из 7512
Обработано адресов: 1500 из 7512
Обработано адресов: 1600 из 7512
Обработано адресов: 1700 из 7512
Обработано адресов: 1800 из 7512
Обработано адресов: 1900 из 7512
Обработано адресов: 2000 из 7512
Обработано адресов: 2100 из 7512
Обработано адресов: 2200 из 7512
Обработано адресов: 2300 из 7512
Обработано адресов: 2400 из 7512
Обработано адресов: 2500 из 7512
Обработано адресов: 2600 из 7512
Обработано адресов: 2700 из 7512
Обработано адресов: 2800 из 7512
Обработан

In [14]:
# возвращаем результат CDI ко всем объектам с таким full_address
address_result = unique_addresses.merge(
    cdi_lookup_df,
    on='address_lookup_id',
    how='left',
    validate='one_to_one',
)

cdi_address_df = sphere_with_row_id.merge(
    address_result.drop(columns=['source_address']),
    left_on='source_address',
    right_on='sphere_full_address',
    how='left',
    validate='many_to_one',
)

for column in [
    'cdi_address_candidate_count',
    'cdi_house_fias_candidate_count',
    'cdi_is_unique_match',
]:
    cdi_address_df[column] = (
        pd.to_numeric(cdi_address_df[column], errors='coerce')
        .fillna(0)
        .astype('int64')
    )

cdi_address_df['cdi_match_status'] = (
    cdi_address_df['cdi_match_status'].fillna('no_source_address')
)
cdi_address_df['cdi_address_match_status'] = cdi_address_df['cdi_match_status']
cdi_address_df['cdi_flat_is_unique_match'] = (
    pd.to_numeric(
        cdi_address_df['cdi_flat_is_unique_match'],
        errors='coerce',
    ).fillna(0).astype('int64')
)
cdi_address_df['cdi_flat_fias_candidate_count'] = (
    pd.to_numeric(
        cdi_address_df['cdi_flat_fias_candidate_count'],
        errors='coerce',
    ).fillna(0).astype('int64')
)
cdi_address_df['cdi_flat_match_status'] = (
    cdi_address_df['cdi_flat_match_status'].fillna('not_found')
)
cdi_address_df['sphere_has_premise_in_address'] = (
    cdi_address_df['source_address']
    .astype('string')
    .str.contains(
        r'(?:^|[,;\s])(?:квартира|кв\.?|офис|помещение|пом\.?|комната|комн\.?|апартамент)',
        case=False,
        regex=True,
        na=False,
    )
)

candidate_rows = cdi_raw_df.copy()

print('Строк после CDI:', len(cdi_address_df))
print(cdi_address_df['cdi_match_status'].value_counts(dropna=False))


Строк после CDI: 14692
cdi_match_status
unique_house_fias    7948
not_found            3785
no_source_address    2959
Name: count, dtype: int64


# 6. Поиск объекта в ЕГРН

Порядок поиска:

1. если CDI вернул ФИАС помещения, ищем точное помещение
2. если ФИАС помещения нет, ищем единственное помещение в доме с такой же площадью
3. если помещение однозначно не найдено, присоединяем только здание

Если остаётся несколько кандидатов, данные ЕГРН не присоединяются.


In [15]:
# готовим ФИАС дома, ФИАС помещения и площадь
egrn_input = cdi_address_df.loc[
    cdi_address_df['cdi_is_unique_match'].eq(1),
    [
        'sphere_row_id',
        'cdi_fias_id_house',
        'cdi_fias_id_flat',
        'total_area',
    ],
].copy()

def json_scalar(value):
    if value is None or pd.isna(value):
        return None
    return str(value)

egrn_records = [
    {
        'sphere_row_id': int(row.sphere_row_id),
        'fias_id_house': json_scalar(row.cdi_fias_id_house),
        'fias_id_flat': json_scalar(row.cdi_fias_id_flat),
        'total_area': json_scalar(row.total_area),
    }
    for row in egrn_input.itertuples(index=False)
]
egrn_json = json.dumps(egrn_records, ensure_ascii=False)

print('Строк передано в поиск ЕГРН:', len(egrn_records))


Строк передано в поиск ЕГРН: 7948


In [16]:
egrn_by_fias_sql = r"""
/* Сначала точное помещение, затем помещение по дому и площади, затем здание. */
with sphere_objects as (
    select /*+ materialize */
        s.sphere_row_id,
        trim(s.fias_id_house) as fias_id_house,
        trim(s.fias_id_flat) as fias_id_flat,
        case
            when regexp_like(replace(trim(s.total_area), ',', '.'), '^[0-9]+([.][0-9]+)?$')
            then to_number(
                replace(trim(s.total_area), ',', '.'),
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as sphere_area
    from json_table(
        :egrn_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            fias_id_house varchar2(500) path '$.fias_id_house',
            fias_id_flat varchar2(500) path '$.fias_id_flat',
            total_area varchar2(200) path '$.total_area'
        )
    ) s
),
raw_candidates as (
    select /*+ no_parallel(e) */
        s.sphere_row_id,
        s.fias_id_house,
        s.fias_id_flat,
        s.sphere_area,
        coalesce(nullif(trim(e.cadaster), ''), 'CAD_IND:' || to_char(e.cad_ind)) as egrn_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        case
            when regexp_like(replace(trim(cast(e.square as varchar2(200))), ',', '.'), '^[0-9]+([.][0-9]+)?$')
            then to_number(
                replace(trim(cast(e.square as varchar2(200))), ',', '.'),
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as egrn_area,
        e.measure,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house as egrn_fias_id_house,
        e.fias_id_flat as egrn_fias_id_flat,
        e.row_update_date,
        e.ias_update_date,
        case when s.fias_id_flat is not null and e.fias_id_flat = s.fias_id_flat then 1 else 0 end as is_flat_fias_match,
        case
            when e.fias_id_flat is not null
              or e.flat is not null or e.flat2 is not null
              or e.office is not null or e.office2 is not null
              or e.room is not null or e.room2 is not null
              or e.compartment1 is not null or e.compartment2 is not null
              or lower(trim(e.oks_type)) like '%помещ%'
            then 1 else 0
        end as is_premise,
        case
            when upper(trim(e.fias_level)) = 'FIAS_HOUSE'
             and lower(trim(e.oks_type)) in ('здание', 'сооружение', 'строение')
             and e.flat is null and e.flat2 is null
             and e.office is null and e.office2 is null
             and e.room is null and e.room2 is null
             and e.compartment1 is null and e.compartment2 is null
            then 1 else 0
        end as is_house
    from sphere_objects s
    join DM_RISK_AVATAR.EGRN_DATA e
      on e.fias_id_house = s.fias_id_house
    where e.cadaster is not null or e.cad_ind is not null
),
ranked as (
    select r.*,
        row_number() over (
            partition by r.sphere_row_id, r.egrn_key
            order by r.row_update_date desc nulls last,
                     r.ias_update_date desc nulls last,
                     r.cad_ind desc nulls last
        ) as version_number
    from raw_candidates r
),
current_rows as (
    select r.*,
        case
            when r.is_premise = 1
             and r.sphere_area > 0
             and r.egrn_area is not null
             and abs(r.egrn_area - r.sphere_area) <= greatest(1, r.sphere_area * 0.01)
            then 1 else 0
        end as is_premise_area_match
    from ranked r
    where r.version_number = 1
),
counts as (
    select
        s.sphere_row_id,
        count(distinct case when r.is_flat_fias_match = 1 then r.egrn_key end) as flat_count,
        count(distinct case when r.is_premise_area_match = 1 then r.egrn_key end) as area_premise_count,
        count(distinct case when r.is_house = 1 then r.egrn_key end) as house_count,
        count(distinct r.egrn_key) as all_count
    from sphere_objects s
    left join current_rows r on r.sphere_row_id = s.sphere_row_id
    group by s.sphere_row_id
),
decision as (
    select c.*,
        case
            when c.flat_count = 1 then 'flat_fias'
            when c.flat_count > 1 then 'ambiguous_flat'
            when c.area_premise_count = 1 then 'house_and_area_premise'
            when c.area_premise_count > 1 then 'ambiguous_premise_area'
            when c.house_count = 1 then 'fias_house_only'
            when c.house_count > 1 then 'ambiguous_house'
            else 'not_found'
        end as match_method
    from counts c
),
chosen as (
    select r.*, d.match_method,
        row_number() over (
            partition by r.sphere_row_id
            order by r.egrn_key
        ) as chosen_number
    from current_rows r
    join decision d on d.sphere_row_id = r.sphere_row_id
    where (d.match_method = 'flat_fias' and r.is_flat_fias_match = 1)
       or (d.match_method = 'house_and_area_premise' and r.is_premise_area_match = 1)
       or (d.match_method = 'fias_house_only' and r.is_house = 1)
)
select /*+ no_parallel */
    s.sphere_row_id as "sphere_row_id",
    s.fias_id_house as "cdi_fias_id_house",
    s.fias_id_flat as "cdi_fias_id_flat",
    d.all_count as "egrn_address_candidate_count",
    case
        when d.match_method = 'flat_fias' then d.flat_count
        when d.match_method = 'house_and_area_premise' then d.area_premise_count
        when d.match_method = 'fias_house_only' then d.house_count
        else greatest(d.flat_count, d.area_premise_count, d.house_count)
    end as "egrn_candidate_count",
    case when d.match_method in ('flat_fias', 'house_and_area_premise', 'fias_house_only') then 1 else 0 end as "egrn_is_unique_match",
    d.match_method as "egrn_match_method",
    case
        when d.match_method in ('flat_fias', 'house_and_area_premise') then 'premise'
        when d.match_method = 'fias_house_only' then 'house'
    end as "egrn_match_level",
    e.cad_ind as "egrn_cad_ind",
    e.cadaster as "egrn_cadaster",
    e.egrn_address as "egrn_address",
    e.square as "egrn_square",
    e.measure as "egrn_measure",
    e.building_type as "egrn_building_type",
    e.oks_type as "egrn_oks_type",
    e.oks_purpose as "egrn_oks_purpose",
    e.object_status as "egrn_object_status",
    e.fias_level as "egrn_fias_level",
    e.egrn_fias_id_house as "egrn_fias_id_house",
    e.egrn_fias_id_flat as "egrn_fias_id_flat"
from sphere_objects s
left join decision d on d.sphere_row_id = s.sphere_row_id
left join chosen e on e.sphere_row_id = s.sphere_row_id and e.chosen_number = 1
order by s.sphere_row_id
"""


In [17]:
egrn_expected_columns = [
    'sphere_row_id', 'cdi_fias_id_house', 'cdi_fias_id_flat',
    'egrn_address_candidate_count', 'egrn_candidate_count',
    'egrn_is_unique_match', 'egrn_match_method', 'egrn_match_level',
    'egrn_cad_ind', 'egrn_cadaster', 'egrn_address', 'egrn_square',
    'egrn_measure', 'egrn_building_type', 'egrn_oks_type',
    'egrn_oks_purpose', 'egrn_object_status', 'egrn_fias_level',
    'egrn_fias_id_house', 'egrn_fias_id_flat',
]

khd_schema = KHD_DATA_SCHEMA.upper()
if not khd_schema.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

if egrn_records:
    egrn_query = egrn_by_fias_sql.replace('DM_RISK_AVATAR.', f'{khd_schema}.')
    with khd_connection.cursor() as cursor:
        bind = cursor.var(oracledb.DB_TYPE_CLOB)
        bind.setvalue(0, egrn_json)
        cursor.execute(egrn_query, egrn_json=bind)
        columns = [str(column[0]).lower() for column in cursor.description]
        rows = cursor.fetchall()
    egrn_lookup_df = pd.DataFrame(rows, columns=columns)
else:
    egrn_lookup_df = pd.DataFrame(columns=egrn_expected_columns)

missing = sorted(set(egrn_expected_columns) - set(egrn_lookup_df.columns))
if missing:
    raise ValueError('ЕГРН не вернул колонки: ' + ', '.join(missing))
if egrn_lookup_df['sphere_row_id'].duplicated().any():
    raise ValueError('ЕГРН вернул несколько итоговых строк для объекта')

expanded_address_egrn_df = cdi_address_df.merge(
    egrn_lookup_df[egrn_expected_columns],
    on='sphere_row_id', how='left', validate='one_to_one',
    suffixes=('', '_egrn_result'),
)
expanded_address_egrn_df['egrn_is_unique_match'] = (
    expanded_address_egrn_df['egrn_is_unique_match'].fillna(0).astype('int64')
)

method_map = {
    'flat_fias': 'egrn_premise_by_flat_fias',
    'house_and_area_premise': 'egrn_premise_by_house_and_area',
    'fias_house_only': 'egrn_house_by_fias',
    'ambiguous_flat': 'egrn_flat_ambiguous',
    'ambiguous_premise_area': 'egrn_premise_area_ambiguous',
    'ambiguous_house': 'egrn_house_ambiguous',
    'not_found': 'cdi_fias_found_egrn_not_found',
}
expanded_address_egrn_df['connection_method'] = (
    expanded_address_egrn_df['egrn_match_method'].map(method_map)
)
expanded_address_egrn_df.loc[
    expanded_address_egrn_df['cdi_match_status'].eq('no_source_address'),
    'connection_method',
] = 'no_source_address'
expanded_address_egrn_df.loc[
    expanded_address_egrn_df['cdi_match_status'].eq('not_found'),
    'connection_method',
] = 'cdi_address_not_found'
expanded_address_egrn_df.loc[
    expanded_address_egrn_df['cdi_match_status'].eq('lookup_error'),
    'connection_method',
] = 'cdi_lookup_error'
expanded_address_egrn_df.loc[
    expanded_address_egrn_df['cdi_match_status'].eq('ambiguous_house_fias'),
    'connection_method',
] = 'cdi_house_fias_ambiguous'

expanded_address_egrn_df['connection_is_unique'] = (
    expanded_address_egrn_df['egrn_is_unique_match'].eq(1).astype('int64')
)
expanded_address_egrn_df['connection_confidence'] = pd.NA
expanded_address_egrn_df.loc[
    expanded_address_egrn_df['egrn_match_method'].eq('flat_fias'),
    'connection_confidence',
] = 'exact'
expanded_address_egrn_df.loc[
    expanded_address_egrn_df['egrn_match_method'].eq('house_and_area_premise'),
    'connection_confidence',
] = 'probable'
expanded_address_egrn_df.loc[
    expanded_address_egrn_df['egrn_match_method'].eq('fias_house_only'),
    'connection_confidence',
] = 'building_only'

if len(expanded_address_egrn_df) != len(expanded_df):
    raise ValueError('После соединения изменилось количество строк')

print('Строк в итоговом датасете:', len(expanded_address_egrn_df))
print(expanded_address_egrn_df['connection_method'].value_counts(dropna=False))


Строк в итоговом датасете: 14692
connection_method
cdi_address_not_found             3785
egrn_house_by_fias                3326
no_source_address                 2959
egrn_house_ambiguous              2827
cdi_fias_found_egrn_not_found      830
egrn_premise_area_ambiguous        468
egrn_premise_by_flat_fias          302
egrn_premise_by_house_and_area     189
egrn_flat_ambiguous                  6
Name: count, dtype: int64


# 7. Проверка результата

Главные колонки:

- `connection_method` — как найден объект;
- `egrn_match_level` — помещение или здание;
- `connection_is_unique` — остался ли один кандидат;
- `connection_confidence` — точная, вероятная или только на уровне здания связь.


In [18]:
quality_profile = pd.DataFrame({
    'Показатель': [
        'Строк в датасете',
        'Точных помещений по ФИАС',
        'Помещений по дому и площади',
        'Связей только со зданием',
        'Неоднозначных связей',
        'Ненайденных связей',
    ],
    'Значение': [
        len(expanded_address_egrn_df),
        expanded_address_egrn_df['connection_method'].eq('egrn_premise_by_flat_fias').sum(),
        expanded_address_egrn_df['connection_method'].eq('egrn_premise_by_house_and_area').sum(),
        expanded_address_egrn_df['connection_method'].eq('egrn_house_by_fias').sum(),
        expanded_address_egrn_df['connection_method'].str.contains('ambiguous', na=False).sum(),
        expanded_address_egrn_df['connection_is_unique'].eq(0).sum(),
    ],
})
display(quality_profile)
display(expanded_address_egrn_df['connection_method'].value_counts(dropna=False))


,Показатель,Значение
0,Строк в датасете,14692
1,Точных помещений по ФИАС,302
2,Помещений по дому и площади,189
3,Связей только со зданием,3326
4,Неоднозначных связей,3301
5,Ненайденных связей,10875


connection_method
cdi_address_not_found             3785
egrn_house_by_fias                3326
no_source_address                 2959
egrn_house_ambiguous              2827
cdi_fias_found_egrn_not_found      830
egrn_premise_area_ambiguous        468
egrn_premise_by_flat_fias          302
egrn_premise_by_house_and_area     189
egrn_flat_ambiguous                  6
Name: count, dtype: int64

# 8. Где теряются объекты

Этот раздел нужен, чтобы не смотреть только на общий процент соединения.

Проверяются переходы:

```text
объект Сферы
→ заполнен full_address
→ CDI вернул один ФИАС дома
→ в ЕГРН нашлись здания с этим ФИАС
→ остался один кандидат
```

Площадь используется только тогда, когда по ФИАС дома нашлось несколько зданий.

In [19]:
diagnostic_df = expanded_address_egrn_df.copy()
reason_summary = (
    diagnostic_df.groupby('connection_method', dropna=False)
    .agg(
        rows=('sphere_row_id', 'size'),
        unique_objects=('object_id', 'nunique'),
        unique_addresses=('source_address', 'nunique'),
        rows_with_area=('total_area', lambda value: pd.to_numeric(value, errors='coerce').gt(0).sum()),
    )
    .reset_index()
    .sort_values('rows', ascending=False)
)
reason_summary['pct_of_all_rows'] = (
    reason_summary['rows'] / len(diagnostic_df) * 100
).round(2)
display(reason_summary)


,connection_method,rows,unique_objects,unique_addresses,rows_with_area,pct_of_all_rows
0,cdi_address_not_found,3785,3784,2467,636,25.76
4,egrn_house_by_fias,3326,3326,2215,260,22.64
8,no_source_address,2959,2957,0,849,20.14
3,egrn_house_ambiguous,2827,2822,1906,304,19.24
1,cdi_fias_found_egrn_not_found,830,830,513,141,5.65
5,egrn_premise_area_ambiguous,468,468,44,468,3.19
6,egrn_premise_by_flat_fias,302,302,274,122,2.06
7,egrn_premise_by_house_and_area,189,187,119,189,1.29
2,egrn_flat_ambiguous,6,6,5,0,0.04


In [20]:
display(reason_summary)


,connection_method,rows,unique_objects,unique_addresses,rows_with_area,pct_of_all_rows
0,cdi_address_not_found,3785,3784,2467,636,25.76
4,egrn_house_by_fias,3326,3326,2215,260,22.64
8,no_source_address,2959,2957,0,849,20.14
3,egrn_house_ambiguous,2827,2822,1906,304,19.24
1,cdi_fias_found_egrn_not_found,830,830,513,141,5.65
5,egrn_premise_area_ambiguous,468,468,44,468,3.19
6,egrn_premise_by_flat_fias,302,302,274,122,2.06
7,egrn_premise_by_house_and_area,189,187,119,189,1.29
2,egrn_flat_ambiguous,6,6,5,0,0.04


In [21]:
display(quality_profile)


,Показатель,Значение
0,Строк в датасете,14692
1,Точных помещений по ФИАС,302
2,Помещений по дому и площади,189
3,Связей только со зданием,3326
4,Неоднозначных связей,3301
5,Ненайденных связей,10875


In [22]:
diagnostic_file = OUTPUT_DIR / 'диагностика_CDI_ЕГРН_помещение.csv'
reason_summary.to_csv(diagnostic_file, index=False, encoding='utf-8-sig')
print('Диагностика:', diagnostic_file)


Диагностика: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\диагностика_CDI_ЕГРН_помещение.csv


# 9. Сохранение результатов


In [23]:
snapshot_at = pd.Timestamp.now(tz='Europe/Moscow').isoformat()
expanded_address_egrn_df['external_snapshot_at'] = snapshot_at

final_path = OUTPUT_DIR / 'датасет_CDI_ЕГРН_помещение_или_здание.csv'
cdi_candidates_path = OUTPUT_DIR / 'снимок_CDI_помещение_или_здание.csv'
cdi_errors_path = OUTPUT_DIR / 'ошибки_CDI_помещение_или_здание.csv'
egrn_path = OUTPUT_DIR / 'снимок_ЕГРН_помещение_или_здание.csv'

expanded_address_egrn_df.to_csv(
    final_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

# сохраняем кандидатов CDI и результат сравнения адресов
candidate_rows.assign(snapshot_at=snapshot_at).to_csv(
    cdi_candidates_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)


cdi_errors_df.assign(snapshot_at=snapshot_at).to_csv(
    cdi_errors_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

egrn_lookup_df.assign(snapshot_at=snapshot_at).to_csv(
    egrn_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

print('Итоговый датасет:', final_path)
print('Кандидаты CDI:', cdi_candidates_path)
print('Ошибки CDI:', cdi_errors_path)
print('Ответы ЕГРН:', egrn_path)


Итоговый датасет: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\датасет_CDI_ЕГРН_помещение_или_здание.csv
Кандидаты CDI: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\снимок_CDI_помещение_или_здание.csv
Ошибки CDI: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\ошибки_CDI_помещение_или_здание.csv
Ответы ЕГРН: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\снимок_ЕГРН_помещение_или_здание.csv


# 10. Список уникальных ИНН

Пустые значения и повторы исключаются.


In [24]:
inn_column = next(
    (
        column
        for column in ['policyholder_inn', 'inn']
        if column in expanded_address_egrn_df.columns
    ),
    None,
)
if inn_column is None:
    raise ValueError('В итоговом датасете не найдена колонка ИНН')

inn_df = (
    expanded_address_egrn_df[[inn_column]]
    .rename(columns={inn_column: 'inn'})
    .assign(inn=lambda frame: frame['inn'].astype('string').str.strip())
    .loc[lambda frame: frame['inn'].notna() & frame['inn'].ne('')]
    .drop_duplicates()
    .sort_values('inn')
    .reset_index(drop=True)
)

inn_path = OUTPUT_DIR / 'inn.csv'
inn_df.to_csv(
    inn_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

print('Уникальных ИНН:', len(inn_df))
print('Файл:', inn_path)


Уникальных ИНН: 476
Файл: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\inn.csv


In [25]:
engine.dispose()
khd_connection.close()
print('Подключения закрыты')


Подключения закрыты
